In [1]:
import tensorflow as tf
from keras.applications import MobileNetV2
from keras.layers import GlobalAveragePooling2D
from keras.models import Model

# 1. Define el tamaño de entrada (con 3 canales)
INPUT_SHAPE = (224, 224, 3)

# 2. Carga el modelo base sin la capa de clasificación
base_model = MobileNetV2(weights='imagenet', 
                         include_top=False, 
                         input_shape=INPUT_SHAPE)

# 3. Congela el modelo base (no queremos re-entrenarlo)
base_model.trainable = False

# 4. Crea tu extractor:
#    Toma la salida del modelo base y aplícale un Pooling
#    Esto convierte el mapa de características (ej. 7x7x1280) 
#    en un solo vector (ej. 1x1280)
x = base_model.output
x = GlobalAveragePooling2D()(x)

# Este es tu modelo extractor final
feature_extractor = Model(inputs=base_model.input, outputs=x)

# Puedes ver la estructura
feature_extractor.summary()

2025-11-21 13:38:46.478857: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-21 13:38:46.488641: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-11-21 13:38:46.500160: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-11-21 13:38:46.503407: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-11-21 13:38:46.511803: I tensorflow/core/platform/cpu_feature_guar

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,257,984 (8.61 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 2,257,984 (8.61 MB)

In [2]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
import os
# Directorio de tu dataset dividido
DATA_DIR = "../dataset_split"
TARGET_SIZE = (224, 224)
BATCH_SIZE = 32

# 1. Crea el generador.
#    OJO: Usamos la función de pre-procesamiento de MobileNetV2
#    El 'color_mode' por defecto es 'rgb', así que cargará
#    tu imagen gris y la duplicará a 3 canales automáticamente.
datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

# 2. Crea los generadores para cada set
train_generator = datagen.flow_from_directory(
    os.path.join(DATA_DIR, 'train'),
    target_size=TARGET_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False  # Importante: no barajar para mantener orden con las etiquetas
)

val_generator = datagen.flow_from_directory(
    os.path.join(DATA_DIR, 'val'),
    target_size=TARGET_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_generator = datagen.flow_from_directory(
    os.path.join(DATA_DIR, 'test'),
    target_size=TARGET_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

Found 2701 images belonging to 3 classes.
Found 1067 images belonging to 3 classes.
Found 560 images belonging to 3 classes.


In [3]:
import numpy as np

print("Extrayendo características del set de entrenamiento...")
features_train = feature_extractor.predict(train_generator)
labels_train = train_generator.classes

print("Extrayendo características del set de validación...")
features_val = feature_extractor.predict(val_generator)
labels_val = val_generator.classes

print("Extrayendo características del set de prueba...")
features_test = feature_extractor.predict(test_generator)
labels_test = test_generator.classes

print(f"Forma de las características de train: {features_train.shape}")
# Salida esperada: (N_imagenes_train, 1280)

Extrayendo características del set de entrenamiento...
85/85 ━━━━━━━━━━━━━━━━━━━━ 25s 283ms/step
Extrayendo características del set de validación...
34/34 ━━━━━━━━━━━━━━━━━━━━ 10s 298ms/step
Extrayendo características del set de prueba...
18/18 ━━━━━━━━━━━━━━━━━━━━ 5s 277ms/step
Forma de las características de train: (2701, 1280)


In [4]:
import pandas as pd
import numpy as np

print("Guardando características en archivos CSV...")

# 1. Obtener el mapeo de clases (para referencia)
#    (Ej: {'COVID': 0, 'Normal': 1, 'Tuberculosis': 2})
class_indices = train_generator.class_indices
print(f"Mapeo de clases (Etiquetas): {class_indices}")

# --- Guardar datos de ENTRENAMIENTO ---
# Crear un DataFrame con las características
df_train = pd.DataFrame(features_train)
# Añadir las etiquetas numéricas como una nueva columna
df_train['label'] = labels_train
# Guardar en CSV
df_train.to_csv('../resultados/train_features.csv', index=False)
print(" - Archivo 'train_features.csv' guardado.")


# --- Guardar datos de VALIDACIÓN ---
df_val = pd.DataFrame(features_val)
df_val['label'] = labels_val
df_val.to_csv('../resultados/val_features.csv', index=False)
print(" - Archivo 'val_features.csv' guardado.")


# --- Guardar datos de PRUEBA ---
df_test = pd.DataFrame(features_test)
df_test['label'] = labels_test
df_test.to_csv('../resultados/test_features.csv', index=False)
print(" - Archivo 'test_features.csv' guardado.")

print("\n¡Archivos CSV generados exitosamente!")

Guardando características en archivos CSV...
Mapeo de clases (Etiquetas): {'covid': 0, 'normal': 1, 'tuberculosis': 2}
 - Archivo 'train_features.csv' guardado.
 - Archivo 'val_features.csv' guardado.
 - Archivo 'test_features.csv' guardado.

¡Archivos CSV generados exitosamente!


In [5]:
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. Modelo SVM (Support Vector Machine)
print("\nEntrenando SVM...")
svm_model = SVC(kernel='rbf', C=1.0) # Kernel RBF es una buena opción por defecto
svm_model.fit(features_train, labels_train)

# 2. Modelo Random Forest
print("Entrenando Random Forest...")
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10)
rf_model.fit(features_train, labels_train)

# 3. Modelo K-Nearest Neighbors
print("Entrenando KNN...")
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(features_train, labels_train)


Entrenando SVM...
Entrenando Random Forest...
Entrenando KNN...


,n_neighbors,5
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'minkowski'
,metric_params,None
,n_jobs,None


In [6]:
# Obtener nombres de las clases para el reporte
class_names = list(train_generator.class_indices.keys())

# Evaluar SVM
print("\n--- Resultados SVM ---")
preds_svm = svm_model.predict(features_test)
print(f"Accuracy (SVM): {accuracy_score(labels_test, preds_svm):.4f}")
print(classification_report(labels_test, preds_svm, target_names=class_names))

# Evaluar Random Forest
print("\n--- Resultados Random Forest ---")
preds_rf = rf_model.predict(features_test)
print(f"Accuracy (RF): {accuracy_score(labels_test, preds_rf):.4f}")
print(classification_report(labels_test, preds_rf, target_names=class_names))

# Evaluar KNN
print("\n--- Resultados KNN ---")
preds_knn = knn_model.predict(features_test)
print(f"Accuracy (KNN): {accuracy_score(labels_test, preds_knn):.4f}")
print(classification_report(labels_test, preds_knn, target_names=class_names))  


--- Resultados SVM ---
Accuracy (SVM): 0.9768
              precision    recall  f1-score   support

       covid       0.95      0.99      0.97       182
      normal       0.99      0.96      0.97       187
tuberculosis       0.99      0.98      0.99       191

    accuracy                           0.98       560
   macro avg       0.98      0.98      0.98       560
weighted avg       0.98      0.98      0.98       560


--- Resultados Random Forest ---
Accuracy (RF): 0.9750
              precision    recall  f1-score   support

       covid       0.97      0.97      0.97       182
      normal       0.98      0.98      0.98       187
tuberculosis       0.97      0.97      0.97       191

    accuracy                           0.97       560
   macro avg       0.97      0.97      0.97       560
weighted avg       0.97      0.97      0.97       560


--- Resultados KNN ---
Accuracy (KNN): 0.9411
              precision    recall  f1-score   support

       covid       0.96      0.95

In [7]:
# --- Celda para verificar sobreajuste ---

print("Evaluando modelos en el set de ENTRENAMIENTO (para comparar)...")

# Evaluar SVM en datos de ENTRENAMIENTO
preds_svm_train = svm_model.predict(features_train)
print(f"\nAccuracy (SVM) en TRAIN: {accuracy_score(labels_train, preds_svm_train):.4f}")
# Evaluar SVM en datos de PRUEBA (ya lo tienes, pero para comparar)
print(f"Accuracy (SVM) en TEST:  {accuracy_score(labels_test, preds_svm):.4f}")

# Evaluar Random Forest en datos de ENTRENAMIENTO
preds_rf_train = rf_model.predict(features_train)
print(f"\nAccuracy (RF) en TRAIN: {accuracy_score(labels_train, preds_rf_train):.4f}")
print(f"Accuracy (RF) en TEST:  {accuracy_score(labels_test, preds_rf):.4f}")

Evaluando modelos en el set de ENTRENAMIENTO (para comparar)...

Accuracy (SVM) en TRAIN: 0.9622
Accuracy (SVM) en TEST:  0.9768

Accuracy (RF) en TRAIN: 0.9970
Accuracy (RF) en TEST:  0.9750


# Implementación de la explicabilidad

## Reconstruir el modelo añadiendo la ultima capa de predicción

In [8]:
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

# 1. Reconstruimos el modelo base
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False 

# 2. Añadimos la cabecera de clasificación
x = base_model.output
x = GlobalAveragePooling2D()(x)
# Capa de salida con 3 neuronas (Covid, Normal, TB) y activación softmax
predictions = Dense(3, activation='softmax')(x)

# 3. Modelo completo end-to-end
gradcam_model = Model(inputs=base_model.input, outputs=predictions)

# 4. Compilar y hacer un entrenamiento RÁPIDO
# Necesitamos esto para que la capa Dense aprenda qué características (pulmones) son importantes
gradcam_model.compile(optimizer=Adam(learning_rate=0.001),
                      loss='categorical_crossentropy',
                      metrics=['accuracy'])

print("Entrenando cabecera para visualización...")
# Usamos los generadores existentes.
gradcam_model.fit(train_generator, epochs=10, validation_data=val_generator)

Entrenando cabecera para visualización...
Epoch 1/10
85/85 ━━━━━━━━━━━━━━━━━━━━ 35s 387ms/step - accuracy: 0.6094 - loss: 1.0043 - val_accuracy: 0.8182 - val_loss: 0.4999
Epoch 2/10
85/85 ━━━━━━━━━━━━━━━━━━━━ 32s 378ms/step - accuracy: 0.7990 - loss: 0.5263 - val_accuracy: 0.8388 - val_loss: 0.4198
Epoch 3/10
85/85 ━━━━━━━━━━━━━━━━━━━━ 32s 378ms/step - accuracy: 0.8393 - loss: 0.4260 - val_accuracy: 0.8922 - val_loss: 0.3411
Epoch 4/10
85/85 ━━━━━━━━━━━━━━━━━━━━ 36s 428ms/step - accuracy: 0.8530 - loss: 0.3907 - val_accuracy: 0.8482 - val_loss: 0.3740
Epoch 5/10
85/85 ━━━━━━━━━━━━━━━━━━━━ 36s 427ms/step - accuracy: 0.8686 - loss: 0.3521 - val_accuracy: 0.9185 - val_loss: 0.2714
Epoch 6/10
85/85 ━━━━━━━━━━━━━━━━━━━━ 33s 393ms/step - accuracy: 0.8923 - loss: 0.3063 - val_accuracy: 0.9044 - val_loss: 0.2799
Epoch 7/10
85/85 ━━━━━━━━━━━━━━━━━━━━ 33s 391ms/step - accuracy: 0.8949 - loss: 0.2951 - val_accuracy: 0.8941 - val_loss: 0.2847
Epoch 8/10
85/85 ━━━━━━━━━━━━━━━━━━━━ 33s 387ms/step - 

## Definición de la función de GRAD-CAM 

In [9]:
import tensorflow as tf
import cv2
import matplotlib.pyplot as plt
import numpy as np

def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    # 1. Crear un modelo que mapee la imagen de entrada a las activaciones de la última capa conv y a las predicciones finales
    grad_model = Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv_layer_name).output, model.output]
    )

    # 2. Grabar las operaciones para el cálculo automático de gradientes
    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]

    # 3. Calcular el gradiente de la clase predicha respecto al mapa de características
    grads = tape.gradient(class_channel, last_conv_layer_output)

    # 4. Global Average Pooling de los gradientes (importancia de cada filtro)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    # 5. Multiplicar cada canal del mapa de características por su importancia
    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    # 6. Aplicar ReLU y normalizar entre 0 y 1
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

def save_and_display_gradcam(img_path, heatmap, cam_path="cam.jpg", alpha=0.4):
    # Cargar imagen original
    img = cv2.imread(img_path)
    img = cv2.resize(img, (224, 224)) # Asegurar tamaño
    
    # Re-escalar mapa de calor al tamaño de imagen
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))

    # Superponer
    superimposed_img = heatmap * alpha + img
    superimposed_img = np.clip(superimposed_img, 0, 255).astype('uint8')

    # Mostrar
    plt.figure(figsize=(10, 4))
    
    plt.subplot(1, 2, 1)
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title("Original")
    plt.axis('off')
    
    plt.subplot(1, 2, 2)
    plt.imshow(cv2.cvtColor(superimposed_img, cv2.COLOR_BGR2RGB))
    plt.title("Grad-CAM")
    plt.axis('off')
    
    plt.show()

## Ejecución con el conjunto de prueba

In [10]:
import multiprocessing
import cv2
import numpy as np
import os
import tensorflow as tf

# --- 1. Definir la función del trabajador (Worker) ---
# (Se mantiene igual que antes)
def guardar_imagen_worker(datos):
    (img_raw, heatmap_raw, true_label, pred_label, confidence, output_dir, file_id) = datos
    
    # A. Des-normalizar
    img_original = ((img_raw + 1) * 127.5).astype('uint8')
    img_original_bgr = cv2.cvtColor(img_original, cv2.COLOR_RGB2BGR)
    
    # B. Procesar Heatmap
    heatmap_resized = cv2.resize(heatmap_raw, (img_original.shape[1], img_original.shape[0]))
    heatmap_resized = np.uint8(255 * heatmap_resized)
    heatmap_colored = cv2.applyColorMap(heatmap_resized, cv2.COLORMAP_JET)
    
    # C. Superponer
    superimposed = cv2.addWeighted(img_original_bgr, 0.6, heatmap_colored, 0.4, 0)
    
    # D. Lado a Lado
    comparative_image = np.hstack([img_original_bgr, superimposed])
    
    # E. Texto
    text = f"Real: {true_label} | Pred: {pred_label} ({confidence:.1%})"
    color_text = (0, 255, 0) if true_label == pred_label else (0, 0, 255)
    cv2.putText(comparative_image, text, (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color_text, 2)
    
    # F. Guardar
    filename = f"{file_id:04d}_{true_label}_vs_{pred_label}.jpg"
    save_path = os.path.join(output_dir, filename)
    cv2.imwrite(save_path, comparative_image)
    
    return filename

# --- 2. Bloque Principal  ---
def procesar_validacion_multiprocessing():
    # CAMBIO 1: Carpeta de salida diferente
    OUTPUT_DIR = "../resultados/comparativas_test_mp"
    if not os.path.exists(OUTPUT_DIR):
        os.makedirs(OUTPUT_DIR)
    
    LAST_CONV_LAYER = "out_relu"
    
    # CAMBIO 2: Usamos test_generator
    class_names = list(test_generator.class_indices.keys())
    test_generator.reset()  # Reiniciar al principio
    
    num_cores = multiprocessing.cpu_count()
    print(f"Iniciando procesamiento de VALIDACIÓN con {num_cores} núcleos...")
    
    pool = multiprocessing.Pool(processes=num_cores)
    
    # CAMBIO 3: steps basado en test_generator
    steps = len(test_generator)
    global_counter = 0
    
    print(f"Generando mapas de calor para {test_generator.n} imágenes de test...")

    try:
        for step in range(steps):
            # Obtener lote de test
            x_batch, y_batch = next(test_generator)
            
            # Predicciones
            preds_batch = gradcam_model.predict(x_batch, verbose=0)
            
            batch_tasks = []
            
            for i in range(len(x_batch)):
                img_array = x_batch[i]
                true_index = np.argmax(y_batch[i])
                true_label = class_names[true_index]
                
                pred_index = np.argmax(preds_batch[i])
                pred_label = class_names[pred_index]
                confidence = np.max(preds_batch[i])
                
                img_exp = np.expand_dims(img_array, axis=0)
                
                # Calcular GradCAM en el proceso principal
                heatmap = make_gradcam_heatmap(img_exp, gradcam_model, LAST_CONV_LAYER, pred_index=pred_index)
                
                task_data = (
                    img_array,
                    heatmap,
                    true_label,
                    pred_label,
                    confidence,
                    OUTPUT_DIR,
                    global_counter
                )
                batch_tasks.append(task_data)
                global_counter += 1
            
            pool.map_async(guardar_imagen_worker, batch_tasks)
            
            print(f"Lote {step+1}/{steps} procesado...", end='\r')
            
    except Exception as e:
        print(f"\nError: {e}")
    finally:
        print("\nEsperando a que terminen los procesos en segundo plano...")
        pool.close()
        pool.join()
        print(f"¡Listo! {global_counter} imágenes de test guardadas en: {OUTPUT_DIR}")

if __name__ == "__main__":
    procesar_validacion_multiprocessing()

Iniciando procesamiento de VALIDACIÓN con 12 núcleos...
Generando mapas de calor para 560 imágenes de test...


/home/merli_nx19/anaconda3/envs/paralelo/lib/python3.10/site-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['keras_tensor_161']
Received: inputs=Tensor(shape=(1, 224, 224, 3))
  warnings.warn(msg)


Lote 18/18 procesado...
Esperando a que terminen los procesos en segundo plano...
¡Listo! 560 imágenes de test guardadas en: ../resultados/comparativas_test_mp


# LISTO LA EXPLICABILIDAD FUNCIONA